# 1. Environment Setup
Install the necessary Hugging Face and PyTorch libraries.

In [1]:
# Install required libraries
!pip install -q transformers datasets torch scikit-learn matplotlib seaborn


# 2. Imports & Data Preprocessing
This block handles data loading and basic cleaning as per the "Mandatory" requirement.


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import BertTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, load_dataset

# 1. DATA PREPROCESSING
# Using a sample of the IMDB dataset for faster execution
dataset = load_dataset('csv', data_files='/content/sample_data/IMDB Dataset.csv', split='train[:2000]')
df = pd.DataFrame(dataset)

# Basic cleaning: Remove missing values
df = df.dropna()

# 2. DATA SPLITTING (Train 70%, Val 15%, Test 15%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# 3. Tokenization
Using the required bert-base-uncased tokenizer.

In [ ]:
# 3. TOKENIZATION
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_func(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Convert to HuggingFace Dataset format
train_ds = Dataset.from_pandas(train_df).map(tokenize_func, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize_func, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_func, batched=True)



# 4. Experiments: Layer Freezing vs. Fine-Tuning
This block sets up the two experiments required by your documentation.

In [ ]:
# Shared training settings
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1}

args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    eval_strategy='epoch'  # Changed from evaluation_strategy
)

# EXPERIMENT 1: Freeze all BERT layers
model_frozen = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
for param in model_frozen.bert.parameters():
    param.requires_grad = False

trainer_frozen = Trainer(model=model_frozen, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer_frozen.train()

# EXPERIMENT 2: Fine-tune last 2 layers
model_ft = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
for param in model_ft.bert.parameters():
    param.requires_grad = False
for layer in model_ft.bert.encoder.layer[-2:]: # Unfreeze last 2
    for param in layer.parameters():
        param.requires_grad = True

trainer_ft = Trainer(model=model_ft, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
trainer_ft.train()


# 5. Final Evaluation
Generate the Confusion Matrix and final metrics.

In [ ]:
# Final Test Evaluation8
preds = trainer_ft.predict(test_ds)
y_preds = preds.predictions.argmax(-1)
y_true = test_df['label']

# Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix: Fine-tuned Model')
plt.show()
print("Final Test Metrics:", preds.metrics)
